# PETA – Photo Album Event Recognition


## Cell 1 - Clone source từ GitHub & kiểm tra môi trường

In [1]:
import subprocess, sys, os, shutil

GITHUB_URL = 'https://github.com/gbao23127157/Final_ComputerVision_PETA.git'
CLONE_DIR  = '/kaggle/working/MyCode'

os.chdir('/kaggle/working')

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

# Clone lại
subprocess.run(['git', 'clone', GITHUB_URL, CLONE_DIR], check=True)

sys.path.insert(0, CLONE_DIR)
os.chdir(CLONE_DIR)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Cloning into '/kaggle/working/MyCode'...


PyTorch : 2.10.0+cu128
GPU     : Tesla T4


In [2]:
%cd /kaggle/working/MyCode/Source

/kaggle/working/MyCode/Source


## Cell 2 - Chuẩn bị cấu trúc thư mục album

In [3]:
IMAGES_DIR  = '/kaggle/input/datasets/huynhtien123/pec-privare/pec/images'
RAW_ALBUMS  = '/kaggle/working/data/raw_albums'
FEATURE_DIR = '/kaggle/working/data/features'
os.makedirs(RAW_ALBUMS,  exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)

# Tạo symlink flat 
for cls_name in os.listdir(IMAGES_DIR):
    cls_dir = os.path.join(IMAGES_DIR, cls_name)
    if not os.path.isdir(cls_dir): continue
    for album_id in os.listdir(cls_dir):
        src = os.path.join(cls_dir, album_id)
        dst = os.path.join(RAW_ALBUMS, f'{cls_name}_{album_id}')
        pt  = os.path.join(FEATURE_DIR, f'{cls_name}_{album_id}.pt')
        if os.path.isdir(src) and not os.path.exists(dst) and not os.path.exists(pt):
            os.symlink(src, dst)

remaining = [
    d for d in os.listdir(RAW_ALBUMS)
    if not os.path.exists(os.path.join(FEATURE_DIR, f'{d}.pt'))
]
print(f'Album cần trích xuất : {len(remaining)}')
print(f'Album đã có .pt      : {len(os.listdir(FEATURE_DIR))}')

Album cần trích xuất : 787
Album đã có .pt      : 17


## Cell 3 - Trích xuất đặc trưng ResNet50

In [4]:
if remaining:
    from extract_features import extract_and_save_features
    extract_and_save_features(RAW_ALBUMS, FEATURE_DIR)
else:
    print('Tất cả features đã có, bỏ qua.')

Đang sử dụng thiết bị tính toán: cuda


Đang trích xuất đặc trưng các album: 100%|██████████| 804/804 [40:52<00:00,  3.05s/it]  


## Cell 4 - Train PETA

In [5]:
os.makedirs('/kaggle/working/Release', exist_ok=True)
os.makedirs('/kaggle/working/Docs',    exist_ok=True)

# Patch 3 đường dẫn trong train.py về /kaggle/working/
with open('train.py') as f:
    src = f.read()

src = src \
    .replace('LOG_PATH = "../Docs/training_log.txt"',
             'LOG_PATH = "/kaggle/working/Docs/training_log.txt"') \
    .replace('SAVE_PATH = "../Release/best_peta_model.pth"',
             'SAVE_PATH = "/kaggle/working/Release/best_peta_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"') \
    .replace('verbose=True', '')

with open('/tmp/train_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/train_kaggle.py

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
2026-04-07 14:15:08 - INFO - Khởi động kịch bản huấn luyện mô hình PETA
2026-04-07 14:15:08 - INFO - Thiết bị sử dụng: cuda
Đã ánh xạ 14 lớp sự kiện: {'birthday': 0, 'children_birthday': 1, 'christmas': 2, 'concert': 3, 'cruise': 4, 'easter': 5, 'exhibition': 6, 'graduation': 7, 'halloween': 8, 'hiking': 9, 'road_trip': 10, 'saint_patricks_day': 11, 'skiing': 12, 'wedding': 13}
2026-04-07 14:15:08 - INFO - Đã chia tập Train gốc | Train: 336 | Val: 84
2026-04-07 14:15:09 - INFO - --- Epoch 1/30 ---


2026-04-07 14:15:21 - INFO - Train - Loss: 1.4781 | Accuracy: 0.5506 | mAP: 0.6330


2026-04-07 14:15:23 - INFO - Val - Loss: 0.8215 | Accuracy: 0.7738 | mAP: 0.9006
2026-04-07 14:15:23 - INFO - Learning Rate hiện tại: 0.0001


2026-04-07 14:15:23 - INFO - -> Đã lưu mô hình tốt nhất với Accuracy: 0.7738 tại /kaggle/working/Release/best_peta_model.pth
2026-04-07 14:15:23 - INFO - --- Epoch 2/30 ---


2026-04-07 14:15:35 - INFO - Train - Loss: 0.6680 | Accuracy: 0.8661 | mAP: 0.9074


...

2026-04-07 14:21:18 - INFO - Val - Loss: 0.4314 | Accuracy: 0.8690 | mAP: 0.9573
2026-04-07 14:21:18 - INFO - Learning Rate hiện tại: 3.125e-06
2026-04-07 14:21:18 - INFO - --- Epoch 29/30 ---


2026-04-07 14:21:29 - INFO - Train - Loss: 0.0245 | Accuracy: 1.0000 | mAP: 1.0000


2026-04-07 14:21:31 - INFO - Val - Loss: 0.4360 | Accuracy: 0.8810 | mAP: 0.9566
2026-04-07 14:21:31 - INFO - Learning Rate hiện tại: 3.125e-06
2026-04-07 14:21:31 - INFO - --- Epoch 30/30 ---


2026-04-07 14:21:42 - INFO - Train - Loss: 0.0189 | Accuracy: 1.0000 | mAP: 1.0000


2026-04-07 14:21:44 - INFO - Val - Loss: 0.4340 | Accuracy: 0.8810 | mAP: 0.9574
2026-04-07 14:21:44 - INFO - Learning Rate hiện tại: 1.5625e-06
2026-04-07 14:21:44 - INFO - Hoàn tất quá trình huấn luyện!


## Cell 5 - Train baseline

In [6]:
os.makedirs('/kaggle/working/Release', exist_ok=True)
os.makedirs('/kaggle/working/Docs',    exist_ok=True)

# Patch đường dẫn trong train_baseline.py về /kaggle/working/
with open('train_baseline.py') as f:
    src = f.read()

src = src \
    .replace('LOG_PATH = "../Docs/training_baseline_log.txt"',
             'LOG_PATH = "/kaggle/working/Docs/training_baseline_log.txt"') \
    .replace('LOG_PATH = "../Docs/training_log.txt"',
             'LOG_PATH = "/kaggle/working/Docs/training_baseline_log.txt"') \
    .replace('SAVE_PATH = "../Release/best_baseline_model.pth"',
             'SAVE_PATH = "/kaggle/working/Release/best_baseline_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"') \
    .replace('verbose=True', '')

with open('/tmp/train_baseline_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/train_baseline_kaggle.py

2026-04-07 14:22:16 - INFO - --- Epoch 1/20 ---


2026-04-07 14:22:17 - INFO - Train - Loss: 2.1234 | Accuracy: 0.3512 | mAP: 0.4216


2026-04-07 14:22:17 - INFO - Val - Loss: 2.2808 | Accuracy: 0.5714 | mAP: 0.7577
2026-04-07 14:22:17 - INFO - -> Đã lưu mô hình tốt nhất với Accuracy: 0.5714 tại /kaggle/working/Release/best_baseline_model.pth
2026-04-07 14:22:17 - INFO - --- Epoch 2/20 ---


2026-04-07 14:22:18 - INFO - Train - Loss: 1.2013 | Accuracy: 0.7351 | mAP: 0.8417


2026-04-07 14:22:18 - INFO - Val - Loss: 1.4819 | Accuracy: 0.7500 | mAP: 0.8440
2026-04-07 14:22:18 - INFO - -> Đã lưu mô hình tốt nhất với Accuracy: 0.7500 tại /kaggle/working/Release/best_baseline_model.pth
2026-04-07 14:22:18 - INFO - --- Epoch 3/20 ---


...

2026-04-07 14:22:33 - INFO - Val - Loss: 0.5455 | Accuracy: 0.8690 | mAP: 0.9033
2026-04-07 14:22:33 - INFO - --- Epoch 19/20 ---


2026-04-07 14:22:34 - INFO - Train - Loss: 0.0990 | Accuracy: 0.9970 | mAP: 1.0000


2026-04-07 14:22:34 - INFO - Val - Loss: 0.5462 | Accuracy: 0.8571 | mAP: 0.9152
2026-04-07 14:22:34 - INFO - --- Epoch 20/20 ---


2026-04-07 14:22:35 - INFO - Train - Loss: 0.0854 | Accuracy: 1.0000 | mAP: 1.0000


2026-04-07 14:22:35 - INFO - Val - Loss: 0.5473 | Accuracy: 0.8571 | mAP: 0.9104


## Cell 6 - Evaluate trên Test set (PETA)

In [7]:
with open('evaluate.py') as f:
    src = f.read()

src = src \
    .replace('MODEL_WEIGHTS = "../Release/best_peta_model.pth"',
             'MODEL_WEIGHTS = "/kaggle/working/Release/best_peta_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"')

with open('/tmp/evaluate_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/evaluate_kaggle.py

Đang chuẩn bị dữ liệu Test...
Đã nạp 140 albums từ tập Test (Dữ liệu hoàn toàn mới).
-> Đã tải thành công file trọng số best_peta_model.pth!


Đang đánh giá mô hình: 100%|██████████| 9/9 [00:02<00:00,  3.66it/s]


 BÁO CÁO KẾT QUẢ TRÊN TẬP TEST (UNSEEN DATA) 
  Accuracy (Độ chính xác) : 0.8786 (87.86%)
  mAP (Mean Average Prec.): 0.9530 (95.30%)


## Cell 7 - Evaluate trên Test set (Baseline)

In [8]:
with open('evaluate_baseline.py') as f:
    src = f.read()

# Patch đường dẫn trọng số của mô hình baseline
src = src \
    .replace('MODEL_WEIGHTS = "../Release/best_baseline_model.pth"',
             'MODEL_WEIGHTS = "/kaggle/working/Release/best_baseline_model.pth"') \
    .replace('FEATURE_DIR = "./data/features"',
             f'FEATURE_DIR = "{FEATURE_DIR}"')

with open('/tmp/evaluate_baseline_kaggle.py', 'w') as f:
    f.write(src)

%run /tmp/evaluate_baseline_kaggle.py

Đang chuẩn bị dữ liệu Test...
-> Đã tải thành công file trọng số /kaggle/working/Release/best_baseline_model.pth!


Đang đánh giá mô hình: 100%|██████████| 9/9 [00:00<00:00, 25.62it/s]


 BÁO CÁO KẾT QUẢ TRÊN TẬP TEST (BASELINE MODEL) 
  Accuracy (Độ chính xác) : 0.8857 (88.57%)
  mAP (Mean Average Prec.): 0.9234 (92.34%)
